In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
GPU name: NVIDIA L4


In [2]:
!nvidia-smi

Sat Oct 25 14:52:11 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:31:00.0 Off |                    0 |
| N/A   37C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset as HFDataset
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
import json
import logging
import random
from datasets import load_dataset, load_from_disk


/home/ec2-user/project/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/ec2-user/project/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Ann

In [4]:
# Hyperparameters

model_name = "HuggingFaceTB/SmolLM-1.7B-Instruct"
dataset_path = "moral_choices"

In [5]:
dataset = load_from_disk(dataset_path)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
  model_name,
    torch_dtype=torch.bfloat16,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [7]:
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

In [8]:
ref_model = AutoModelForCausalLM.from_pretrained(
  model_name,
    torch_dtype=torch.bfloat16,
).eval()

In [9]:
	peft_config = LoraConfig(
		lora_alpha=64,
		lora_dropout=0.1,
		r=32,
		bias="none",
		task_type="CAUSAL_LM",
		target_modules="all-linear"
	)

In [14]:
import wandb
wandb.init(project="dpo-training", name="1.7B-lora-dpo-100k")

In [15]:
training_args = DPOConfig(
    num_train_epochs=4,
    learning_rate=5e-05,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    do_eval=True,
    per_device_eval_batch_size=64,
    adam_epsilon=1e-08,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    seed=42,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2, 
    eval_strategy="steps",
    eval_steps=600,
    output_dir="./output-dir",
    gradient_checkpointing=True,
    bf16=True,
    remove_unused_columns=False,
    report_to="wandb",
    beta=0.1,
    loss_type="sigmoid",
    max_grad_norm=1.0,
    max_length=4096,  # or whatever covers your prompts + ~10 tokens
    max_prompt_length=4000,  # leaves room for the short completion
)

In [16]:
dpo_trainer = DPOTrainer(
model=model,
ref_model=None,
    args=training_args,
    # beta=training_args.beta,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
)

/home/ec2-user/project/.venv/lib/python3.12/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/ec2-user/project/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
dpo_trainer.train()

/home/ec2-user/project/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
600,0.518900,0.520994,-0.693981,-1.515637,0.769307,0.821656,-22.117022,-30.295710,0.064855,0.245960


In [ ]:
i = 5